# Reproducing the study, step by step

This notebook re-derives the study's headline results by hand — from the raw calculation
records to the numbers in the manuscript. It is the companion to the paper's Results:
every number below is computed here, in the open, and you can trace each one back to
the record it came from.

The repo has three parts, and this notebook is the bridge between the first two:

- **Evidence** — `data/calculations/`: 420 calculation records (geometry, frequencies, energies)
- **Claims** — `data/published/`: the manuscript's tables, as published
- **Derivation** — this notebook: evidence → claims, written out

(`atcgu reproduce` does the same derivation through the tested package and checks all
28 published tables automatically. Here we redo the key steps ourselves so you can see
how each number arises.)

**How to use it:** run the cells top to bottom. The five steps follow the study:

1. The data — what a calculation record is
2. Baseline reaction energies across the 21 families
3. Sensitivity across four functional/solvation treatments
4. The two-water placement test
5. Decomposing the net reaction


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Find the repo root (works from the checkout root or a subdirectory)
ROOT = Path().resolve()
while not (ROOT / "config" / "molecules.csv").exists():
    ROOT = ROOT.parent
    assert str(ROOT) != "/", "could not find the repo checkout"

KCAL = json.loads((ROOT / "config/study.json").read_text())["hartree_to_kcal_mol"]
print("repo root:", ROOT)
print("hartree -> kcal/mol:", KCAL)


repo root: /home/hatch/workspace/repo-audit
hartree -> kcal/mol: 627.5094740631


## Step 1 — The data: what a calculation record is

Each of the 420 records in `data/calculations/` bundles everything one quantum-chemistry
calculation produced: the starting and optimized geometries, every vibrational frequency,
and the energies. Three records document failed source calculations; the rest are usable.


In [2]:
manifest = pd.read_csv(ROOT / "data/calculations/manifest.csv")
print(f"{len(manifest)} calculation records")
print(manifest["status"].value_counts().to_string())
print()
print(manifest[["calculation_id", "status"]].head(3).to_string(index=False))


420 calculation records
status
valid     417
failed      3

       calculation_id status
calc_557d5df31f5d3868  valid
calc_bdef5ac53c00d7a1  valid
calc_8fff6d635cf75f67  valid


Pick one record — the selected adenine nucleotide at the baseline level — and look
inside. The Gibbs free energy we use everywhere is just:

**G = E(electronic) + G(thermal correction)**

both printed by the frequency job and stored in the record.


In [3]:
# Which record was selected for the adenine nucleotide at baseline?
sel5 = pd.read_csv(ROOT / "data/published/continuum/05_selected_species.csv")
base = sel5[sel5["analysis_id"] == "full21_b3lyp_pcm"]
calc_id = base.loc[base["species_id"] == "adenine_ribose_phosphate", "calculation_id"].iloc[0]
rec_path = manifest.loc[manifest["calculation_id"] == calc_id, "record_path"].iloc[0]
rec = json.loads((ROOT / rec_path).read_text())

E = rec["electronic_energy_hartree"]
corr = rec["thermal_gibbs_correction_hartree"]
G = rec["computed_gibbs_hartree"]
print(f"record:            {calc_id}")
print(f"E(electronic):    {E:.6f} hartree")
print(f"G correction:     {corr:.6f} hartree")
print(f"E + correction:   {E + corr:.6f} hartree")
print(f"stored G:         {G:.6f} hartree")
assert abs((E + corr) - G) < 1e-9, "G must equal E + thermal correction"
print("-> G = E + correction checks out")
print(f"vibrational modes stored: {rec['frequency_count']} "
      f"(imaginary: {rec['imaginary_frequency_count']})")


record:            calc_565406d4de58dfbc
E(electronic):    -1531.789792 hartree
G correction:     0.223677 hartree
E + correction:   -1531.566115 hartree
stored G:         -1531.566115 hartree
-> G = E + correction checks out
vibrational modes stored: 105 (imaginary: 0)


Why *this* record for the adenine nucleotide? Conformational sampling produced
several candidates per species; the study keeps the **lowest valid sampled Gibbs energy**
at 298 K. Here are adenine nucleotide's candidates — the selected one is simply the
minimum:


In [4]:
cand = pd.read_csv(ROOT / "data/published/continuum/04_analysis_candidates.csv")
a = cand[(cand["analysis_id"] == "full21_b3lyp_pcm")
         & (cand["species_id"] == "adenine_ribose_phosphate")].copy()
a = a.sort_values("computed_gibbs_hartree").reset_index(drop=True)
a["kcal_from_min"] = (a["computed_gibbs_hartree"] - a["computed_gibbs_hartree"].min()) * KCAL
print(a[["manifest_file_name", "computed_gibbs_hartree", "kcal_from_min"]].to_string(index=False))
print(f"\n{a['species_id'].iloc[0]}: {len(a)} candidates, lowest selected "
      f"(next is +{a['kcal_from_min'].iloc[1]:.2f} kcal/mol)")


                             manifest_file_name  computed_gibbs_hartree  kcal_from_min
  adenine_ribose_phosphate_conf13_pcm_water.log            -1531.566115       0.000000
   adenine_ribose_phosphate_conf1_pcm_water.log            -1531.561955       2.610182
  adenine_ribose_phosphate_conf22_pcm_water.log            -1531.561742       2.744074
  adenine_ribose_phosphate_conf75_pcm_water.log            -1531.560810       3.328674
 adenine_ribose_phosphate_conf113_pcm_water.log            -1531.558909       4.521708
adenine_ribose_phosphate_dft_seed_pcm_water.log            -1531.556582       5.981552

adenine_ribose_phosphate: 6 candidates, lowest selected (next is +2.61 kcal/mol)


## Step 2 — Baseline: the net reaction across 21 families

The study's core comparison is one balanced net reaction per base family:

**base + ribose + phosphoric acid → nucleotide + 2 H₂O**

with every species as a neutral singlet. For family *i*:

**ΔGᵢ = G(nucleotideᵢ) + 2·G(water) − G(baseᵢ) − G(ribose) − G(phosphoric acid)**

and everything is reported relative to adenine: **ΔΔGᵢ = ΔGᵢ − ΔG(adenine)**.
Negative ΔΔG means more favorable formation than adenine. Let's compute it —
per family, from the selected species Gibbs energies, by hand:


In [5]:
fams = pd.read_csv(ROOT / "config/reaction_families.csv")
G = dict(zip(base["species_id"], base["computed_gibbs_hartree"]))  # hartree

rows = []
for f in fams.to_dict("records"):
    dG = (G[f["nucleotide_species_id"]] + 2 * G["water"]
          - G[f["base_species_id"]] - G["ribose"] - G["phosphate"]) * KCAL
    rows.append({"family": f["display_name"],
                 "canonical": str(f["canonical"]) == "True",
                 "dG_rxn": dG})
rxn = pd.DataFrame(rows)
dG_ade = rxn.loc[rxn["family"] == "Adenine", "dG_rxn"].iloc[0]
rxn["ddG_vs_adenine"] = rxn["dG_rxn"] - dG_ade
rxn = rxn.sort_values("ddG_vs_adenine").reset_index(drop=True)
rxn["rank"] = rxn.index + 1
print(rxn[["rank", "family", "ddG_vs_adenine"]].to_string(index=False, float_format="%.2f"))


 rank                         family  ddG_vs_adenine
    1                        Adenine            0.00
    2                       Cytosine            0.36
    3                       Melamine            3.70
    4                      Imidazole            4.39
    5          2,4,6-TAP (C5-linked)            4.88
    6                        Guanine            5.16
    7 5-Amino-4-imidazolecarboxamide            5.36
    8                   Hypoxanthine            6.00
    9     Barbituric acid (C-linked)            6.13
   10                  Dihydrouracil            6.58
   11                         Purine            6.68
   12                        Thymine            7.87
   13                         Uracil            7.90
   14              2,6-Diaminopurine            8.14
   15    Imidazole-2-carboxylic acid            8.59
   16       Cyanuric acid (N-linked)            9.08
   17                       Xanthine            9.23
   18     Barbituric acid (N-linked)          

We just re-derived the pipeline's central table. Check it against the published one —
the largest difference should be numerical noise:


In [6]:
pub = pd.read_csv(ROOT / "data/published/continuum/09_relative_rankings.csv")
pub = pub[(pub["panel"] == "all_21")
          & (pub["reaction_id"] == "P1_total_net_nucleotide_formation")]
pub = pub.drop_duplicates("family_display_name")[["family_display_name", "relative_g_kcal_mol"]]
merged = rxn.merge(pub, left_on="family", right_on="family_display_name")
diff = (merged["ddG_vs_adenine"] - merged["relative_g_kcal_mol"]).abs().max()
print(f"families compared: {len(merged)}")
print(f"max |hand-derived - published| = {diff:.2e} kcal/mol")
assert diff < 1e-9
print("-> our derivation reproduces the published baseline table")


families compared: 21
max |hand-derived - published| = 2.85e-10 kcal/mol
-> our derivation reproduces the published baseline table


And the headline numbers from the manuscript's Results, straight from our table:


In [7]:
vals = rxn["ddG_vs_adenine"].tolist()
names = rxn["family"].tolist()
print(f"A/C separation: {vals[1] - vals[0]:.2f} kcal/mol  (manuscript: 0.36)")
print(f"rank 12 / 13:   {names[11]} / {names[12]}")
print(f"order:          {', '.join(names[:6])} ...")


A/C separation: 0.36 kcal/mol  (manuscript: 0.36)
rank 12 / 13:   Thymine / Uracil
order:          Adenine, Cytosine, Melamine, Imidazole, 2,4,6-TAP (C5-linked), Guanine ...


## Step 3 — Sensitivity: four functional/solvation treatments

The same net reaction was recomputed for ten families under four combinations of
electronic-structure method and implicit solvation. Same derivation, grouped by
treatment — do the rankings survive?


In [8]:
treatments = {
    "B3LYP/PCM":   "method_sensitivity10_1_b3lyp_pcm",
    "B3LYP/SMD":   "method_sensitivity10_2_b3lyp_smd",
    "wB97X-D/PCM": "method_sensitivity10_3_wb97xd_pcm",
    "wB97X-D/SMD": "method_sensitivity10_4_wb97xd_smd",
}
fam_of = {}
for f in fams.to_dict("records"):
    for sid in (f["base_species_id"], f["nucleotide_species_id"]):
        fam_of[sid] = f["display_name"]

sens_rows = []
for label, aid in treatments.items():
    s = sel5[sel5["analysis_id"] == aid]
    Gs = dict(zip(s["species_id"], s["computed_gibbs_hartree"]))
    for f in fams.to_dict("records"):
        b, n = f["base_species_id"], f["nucleotide_species_id"]
        if b not in Gs or n not in Gs:
            continue  # family not in this ten-family subset
        dG = (Gs[n] + 2 * Gs["water"] - Gs[b] - Gs["ribose"] - Gs["phosphate"]) * KCAL
        sens_rows.append({"treatment": label, "family": f["display_name"], "dG": dG})

sens = pd.DataFrame(sens_rows)
sens["ddG"] = sens["dG"] - sens.groupby("treatment")["dG"].transform(
    lambda s: s[sens.loc[s.index, "family"] == "Adenine"].iloc[0])
pivot = sens.pivot(index="family", columns="treatment", values="ddG").round(2)
print(pivot.to_string())

mel = sens[sens["family"] == "Melamine"]["ddG"]
cyt = sens[sens["family"] == "Cytosine"]["ddG"]
print(f"\nmelamine spread: {mel.min():.2f} to {mel.max():.2f} kcal/mol")
print(f"cytosine range:   {cyt.min():.2f} to {cyt.max():.2f} kcal/mol")
for label in treatments:
    top2 = sens[sens["treatment"] == label].nsmallest(2, "ddG")["family"].tolist()
    print(f"{label:12s} two lowest: {top2[0]}, {top2[1]}")


treatment                   B3LYP/PCM  B3LYP/SMD  wB97X-D/PCM  wB97X-D/SMD
family                                                                    
2,4,6-TAP (C5-linked)            4.88       0.99         4.61         1.64
Adenine                          0.00       0.00         0.00         0.00
Barbituric acid (C-linked)       6.13       2.80         5.42         2.78
Cytosine                         0.36      -0.92        -0.30         0.40
Guanine                          5.16       2.17         5.96         4.90
Imidazole                        4.39       2.15         6.28         3.23
Melamine                         3.70       1.15         7.53         4.47
Thymine                          7.87       4.84         7.49         6.15
Uracil                           7.90       5.69         9.90         7.44
Xanthine                         9.23       4.03         8.39         4.52

melamine spread: 1.15 to 7.53 kcal/mol
cytosine range:   -0.92 to 0.40 kcal/mol
B3LYP/PCM    two lo

## Step 4 — The two-water placement test

Beyond implicit solvation: two explicit waters were added to each of the six
leading nucleotide products, in two starting arrangements per family. How much do
the two optimized placements disagree, and does the A/C lead survive the
lower-energy choice?


In [9]:
si = pd.read_csv(ROOT / "data/published/si_microsolvation/05_fixed_reference_placement_scores.csv")
h = si[si["treatment"] == "harmonic"].copy()

pretty = {"adenine": "Adenine", "cytosine": "Cytosine", "guanine": "Guanine",
          "imidazole": "Imidazole", "melamine": "Melamine", "tap": "TAP (C5-linked)"}
diffs = h.groupby("family")["fixed_reference_score_kcal_mol"].agg(lambda s: s.max() - s.min())
print("placement disagreement per family (kcal/mol):")
for fam, d in diffs.items():
    print(f"  {pretty[fam]:15s} {d:.2f}")
print(f"\nrange: {diffs.min():.2f} - {diffs.max():.2f} kcal/mol  (manuscript: 0.43-3.82)")

chosen = h[h["selected_for_family"]].sort_values("fixed_reference_score_kcal_mol")
print("\nselected (lower-energy) placement order:")
print("  " + ", ".join(pretty[f] for f in chosen["family"]))


placement disagreement per family (kcal/mol):
  Adenine         3.82
  Cytosine        2.18
  Guanine         2.30
  Imidazole       0.90
  Melamine        1.72
  TAP (C5-linked) 0.43

range: 0.43 - 3.82 kcal/mol  (manuscript: 0.43-3.82)

selected (lower-energy) placement order:
  Cytosine, Adenine, Imidazole, Melamine, TAP (C5-linked), Guanine


And under the alternative low-frequency entropy treatments (50/100 cm⁻¹ floors,
Grimme qRRHO), re-selecting the lower placement per family per treatment:


In [10]:
summ = pd.read_csv(ROOT / "data/published/si_microsolvation/thermochemical_summary.csv")
print(summ[["treatment", "selected_order"]].to_string(index=False))
ca = summ["c_minus_a"].abs()
print(f"\ncytosine-adenine separation across treatments: {ca.min():.3f} to {ca.max():.3f} kcal/mol")
print("-> the six-family order is preserved under every entropy treatment")


       treatment                                            selected_order
       floor_100 cytosine < adenine < imidazole < melamine < tap < guanine
        floor_50 cytosine < adenine < imidazole < melamine < tap < guanine
grimme_qrrho_100 cytosine < adenine < imidazole < melamine < tap < guanine
        harmonic cytosine < adenine < imidazole < melamine < tap < guanine

cytosine-adenine separation across treatments: 0.011 to 0.442 kcal/mol
-> the six-family order is preserved under every entropy treatment


## Step 5 — Decomposing the net reaction

The net reaction splits into nucleoside formation and phosphorylation:

**base + ribose → nucleoside + H₂O**  →  **nucleoside + phosphoric acid → nucleotide + H₂O**

A base that looks good in one step can lose it in the other. Rank the families in
each step separately:


In [11]:
r09 = pd.read_csv(ROOT / "data/published/continuum/09_relative_rankings.csv")
p = r09[r09["panel"] == "all_21"]
for rxn_id, label in [("P1a_nucleoside_formation", "nucleoside formation"),
                      ("P1b_nucleotide_from_nucleoside", "phosphorylation")]:
    t = (p[p["reaction_id"] == rxn_id]
         .drop_duplicates("family_display_name")
         .sort_values("relative_g_kcal_mol").reset_index(drop=True))
    t["rank"] = t.index + 1
    top = t.head(3)[["rank", "family_display_name"]].to_string(index=False)
    dap = t.loc[t["family_display_name"] == "2,6-Diaminopurine", "rank"].iloc[0]
    print(f"{label}:" + chr(10) + top + chr(10) + f"  ... 2,6-diaminopurine: rank {dap} of {len(t)}" + chr(10))


nucleoside formation:
 rank family_display_name
    1             Adenine
    2   2,6-Diaminopurine
    3            Cytosine
  ... 2,6-diaminopurine: rank 2 of 21

phosphorylation:
 rank family_display_name
    1            Melamine
    2         Isocytosine
    3            Cytosine
  ... 2,6-diaminopurine: rank 21 of 21



That's the manuscript's point in one picture: 2,6-diaminopurine is 2nd at making
the nucleoside and 21st (last) at phosphorylating it — netting out at 14th overall.
Melamine is the reverse: middling at the first step, best at the second. Whether a
base looks "favored" depends on which transformation you ask about.

---

**Where every manuscript number came from:**

| Manuscript claim | Derived in |
|---|---|
| A/C lead, 0.36 separation; T/U at 12/13 | Step 2 |
| Melamine 1.15–7.53 spread; cytosine −0.92/+0.40; A/C top-2 everywhere | Step 3 |
| Placement diffs 0.43–3.82; selected order C, A, imidazole, melamine, TAP, G | Step 4 |
| Entropy treatments preserve order; A/C sep 0.011–0.442 | Step 4 |
| DAP 2nd / 21st; melamine 1st at phosphorylation | Step 5 |

For the full machine-checked verification — all 28 published tables, not just the
headlines — run `atcgu reproduce --out outputs/my_review` and read
`VALIDATION_REPORT.md`.


In [12]:
from IPython.display import Image
Image(ROOT / "results/manuscript/figure1.png", width=880)
